# Task 3 - SARIMA (Kaggle/Colab)
Run SARIMA for the three required areas and save predictions + metrics.

In [ ]:
# Environment-aware setup + data loading
# This cell supports both Kaggle and Colab and resolves the parquet path automatically.
from pathlib import Path
import os
import time
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
CITY_FILE = "city_traffic.parquet"

if IS_COLAB:
    # Mount Google Drive so the notebook can read files from MyDrive.
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

def resolve_data_path() -> Path:
    # Kaggle: search attached datasets under /kaggle/input.
    if IS_KAGGLE:
        matches = list(Path("/kaggle/input").rglob(CITY_FILE))
        if matches:
            return matches[0]

    # Colab: user said parquet is under MyDrive/Colab Notebooks/.
    if IS_COLAB:
        colab_root = Path("/content/drive/MyDrive/Colab Notebooks")
        direct = colab_root / CITY_FILE
        if direct.exists():
            return direct

        # Fallback search if parquet is inside a subfolder.
        matches = list(colab_root.rglob(CITY_FILE))
        if matches:
            return matches[0]

    raise FileNotFoundError(
        "city_traffic.parquet not found. ",
Kaggle: attach dataset first. ",
Colab: upload under MyDrive/Colab Notebooks."
    )

DATA_PATH = resolve_data_path()
print(f"Using dataset: {DATA_PATH}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)

# Normalize common column name variants so the rest of the notebook is stable.
rename_map = {
    "squareid": "square_id",
    "square": "square_id",
    "time": "time_interval",
    "timestamp": "time_interval",
    "internet": "internet_traffic",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

required_cols = ["square_id", "time_interval", "internet_traffic"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}. Available: {df.columns.tolist()}")

df["time_interval"] = pd.to_datetime(df["time_interval"], utc=True)
df = df.sort_values(["square_id", "time_interval"]).reset_index(drop=True)

def get_area_series(data: pd.DataFrame, square_id: int) -> pd.Series:
    s = data[data["square_id"] == square_id].sort_values("time_interval")
    series = s.set_index("time_interval")["internet_traffic"].astype(float)

    # Enforce fixed 10-minute frequency to avoid statsmodels date-index warnings.
    series = series.resample("10min").mean()
    series = series.interpolate(limit_direction="both")
    return series

def split_train_test(series: pd.Series, fast_mode: bool = True):
    test_start = pd.Timestamp("2013-12-16 00:00:00", tz="UTC")
    test_end = pd.Timestamp("2013-12-22 23:50:00", tz="UTC")
    if fast_mode:
        # Keep only a recent window so SARIMA runs faster in Colab.
        train_start = test_start - pd.Timedelta(days=30)
        train_end = test_start - pd.Timedelta(minutes=10)
        train = series.loc[train_start:train_end]
    else:
        train_end = pd.Timestamp("2013-12-15 23:50:00", tz="UTC")
        train = series.loc[:train_end]

    test = series.loc[test_start:test_end]
    return train.dropna(), test.dropna()

top_square = int(df.groupby("square_id")["internet_traffic"].sum().idxmax())
target_squares = [top_square, 4159, 4556]
print(f"Target squares: {target_squares}")

In [ ]:
# SARIMA training + evaluation
# Fast-mode defaults are tuned for deadline-friendly execution while keeping outputs consistent.
def evaluate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = float(np.mean(np.abs(y_true - y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return {"MAE": mae, "MAPE": mape, "RMSE": rmse}

FAST_MODE = True
ORDER = (1, 1, 1)
SEASONAL_ORDER = (1, 0, 1, 144)
MAXITER = 40

sarima_results = []
for square_id in target_squares:
    series = get_area_series(df, int(square_id))
    train, test = split_train_test(series, fast_mode=FAST_MODE)

    if len(train) < 3 * 144 or len(test) == 0:
        print(f"Skipping square {square_id}: insufficient train/test points")
        continue

    start = time.perf_counter()
    model = SARIMAX(
        train,
        order=ORDER,
        seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fit = model.fit(disp=False, maxiter=MAXITER)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    forecast = fit.forecast(steps=len(test))
    infer_time = time.perf_counter() - start

    y_true = test.values
    y_pred = forecast.values
    metrics = evaluate_metrics(y_true, y_pred)
    metrics.update({"square_id": int(square_id), "train_seconds": train_time, "infer_seconds": infer_time})
    sarima_results.append(metrics)
    print(f"Square {square_id} done in {train_time:.1f}s (train), {infer_time:.2f}s (forecast)")

    out = pd.DataFrame({
        "time_interval": test.index,
        "y_true": y_true,
        "y_pred": y_pred,
        "model": "SARIMA",
        "square_id": int(square_id),
    })
    out.to_csv(OUTPUT_DIR / f"sarima_{int(square_id)}.csv", index=False)

pd.DataFrame(sarima_results).to_csv(OUTPUT_DIR / "metrics_sarima.csv", index=False)
pd.DataFrame(sarima_results)